In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader
import torch.optim as optim
import os
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image
from torch.optim.lr_scheduler import ReduceLROnPlateau


In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [3]:
class CrackDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.labels_df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform
        self.label_map = {'Non-cracked': 0, 'Cracked': 1}

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        img_name = self.labels_df.iloc[idx, 0]
        label = self.label_map[self.labels_df.iloc[idx, 1]]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [4]:
# Load datasets
train_dataset = CrackDataset('../artifact_folder/train/labels.csv', '../artifact_folder/train/images', transform)
val_dataset = CrackDataset('../artifact_folder/val/labels.csv', '../artifact_folder/val/images', transform)
test_dataset = CrackDataset('../artifact_folder/test/labels.csv', '../artifact_folder/test/images', transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)


In [5]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)  # 2 classes: Cracked / Non-cracked


/Users/nanphattongsirisukool/Documents/GitHub/Structural-Defects-Network-MLOps/env/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/nanphattongsirisukool/Documents/GitHub/Structural-Defects-Network-MLOps/env/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [7]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()

        train_acc = correct / len(train_loader.dataset)
        print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}, Accuracy: {train_acc:.4f}")


In [8]:
def evaluate(model, val_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
    val_accuracy = correct / len(val_loader.dataset)
    print(f'Validation Accuracy: {val_accuracy:.4f}')


In [9]:
train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10)
evaluate(model, val_loader)


Epoch 1, Loss: 9484.7536, Accuracy: 0.7099
Epoch 2, Loss: 7863.7744, Accuracy: 0.7823
Epoch 3, Loss: 7252.5389, Accuracy: 0.7979
Epoch 4, Loss: 6645.8552, Accuracy: 0.8247
Epoch 5, Loss: 6229.9283, Accuracy: 0.8363
Epoch 6, Loss: 5826.3927, Accuracy: 0.8533
Epoch 7, Loss: 4860.3041, Accuracy: 0.8767
Epoch 8, Loss: 4357.0149, Accuracy: 0.8927
Epoch 9, Loss: 3450.0728, Accuracy: 0.9190
Epoch 10, Loss: 2754.8989, Accuracy: 0.9380
Validation Accuracy: 0.8285


In [ ]:
torch.save(model.state_dict(), '../model/resnet_crack_classifier_new_folder.pth')

In [10]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

# Make sure model is in eval mode
model.eval()

# Consistent transform (same used during training)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Evaluation function
def evaluate_model_on_split(split_name):
    base_path = f"../artifact_folder/{split_name}"
    labels_df = pd.read_csv(os.path.join(base_path, "labels.csv"))
    images_dir = os.path.join(base_path, "images")

    y_true, y_pred, y_score = [], [], []

    for _, row in labels_df.iterrows():
        img_path = os.path.join(images_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")
        input_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            pred = output.argmax(dim=1).item()
            prob = torch.softmax(output, dim=1)[0][1].item()

        label = 1 if row["label"].lower() == "cracked" else 0
        y_true.append(label)
        y_pred.append(pred)
        y_score.append(prob)

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "AUC-ROC": roc_auc_score(y_true, y_score),
        "Confusion Matrix": confusion_matrix(y_true, y_pred).tolist()
    }

# Run on all sets
for split in ['train', 'val', 'test']:
    print(f"📊 Evaluation for {split.upper()}")
    metrics = evaluate_model_on_split(split)
    for k, v in metrics.items():
        print(f"{k}: {v}")
    print()


📊 Evaluation for TRAIN
Accuracy: 0.9598656294200849
Precision: 0.9441092771770062
Recall: 0.9776049033474776
F1-Score: 0.9605651745903063
AUC-ROC: 0.9944339996234419
Confusion Matrix: [[7993, 491], [190, 8294]]

📊 Evaluation for VAL
Accuracy: 0.8284797337453941
Precision: 0.46492374727668845
Recall: 0.8322932917316692
F1-Score: 0.5965893206597708
AUC-ROC: 0.9140059081538694
Confusion Matrix: [[5903, 1228], [215, 1067]]

📊 Evaluation for TEST
Accuracy: 0.8312537136066548
Precision: 0.46578366445916114
Recall: 0.83399209486166
F1-Score: 0.5977337110481586
AUC-ROC: 0.9096191160618039
Confusion Matrix: [[5940, 1210], [210, 1055]]



In [7]:
class SoftF1Loss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)[:, 1]
        targets = targets.float()

        TP = (probs * targets).sum()
        FP = (probs * (1 - targets)).sum()
        FN = ((1 - probs) * targets).sum()

        soft_f1 = 2 * TP / (2 * TP + FP + FN + 1e-8)
        return 1 - soft_f1  # we want to minimize loss, so use 1 - F1


In [8]:
from sklearn.metrics import f1_score

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler=None, epochs=10):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = correct / len(train_loader.dataset)

        # Evaluate on validation set
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)

                preds = outputs.argmax(1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_f1 = f1_score(all_labels, all_preds)

        if scheduler:
            scheduler.step(val_loss)  # Optionally use val_loss or convert to custom scheduler

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")


In [9]:
import optuna

def objective(trial):
    # Hyperparameter suggestions
    lr = trial.suggest_loguniform('lr', 1e-5, 1e-3)
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)

    # Model
    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.to(device)

    criterion = SoftF1Loss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Train 1–3 epochs quickly for evaluation
    train_model(model, train_loader, val_loader, criterion, optimizer, epochs=3)

    # After training, evaluate F1 on val set
    f1 = evaluate_model_on_split('val')["F1-Score"]
    return f1


/Users/nanphattongsirisukool/Documents/GitHub/Structural-Defects-Network-MLOps/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print("Best trial:")
print(study.best_trial)


[I 2025-05-18 04:16:30,467] A new study created in memory with name: no-name-a8b1fb07-b4e8-4881-ba37-19d46bf86e50


/var/folders/4g/8j_z4q0n2zlftk4s8jmrj1sw0000gn/T/ipykernel_36442/2510586555.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform('lr', 1e-5, 1e-3)
/var/folders/4g/8j_z4q0n2zlftk4s8jmrj1sw0000gn/T/ipykernel_36442/2510586555.py:6: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
/Users/nanphattongsirisukool/Documents/GitHub/Structural-Defects-Network-MLOps/env/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/nanphat

Epoch 1/3 | Train Loss: 0.3292 | Train Acc: 0.6763 | Val Loss: 0.4768 | Val F1: 0.5725
Epoch 2/3 | Train Loss: 0.2116 | Train Acc: 0.8040 | Val Loss: 0.3903 | Val F1: 0.6499
Epoch 3/3 | Train Loss: 0.1687 | Train Acc: 0.8447 | Val Loss: 0.3204 | Val F1: 0.7164


[I 2025-05-18 05:38:42,905] Trial 0 finished with value: 0.07372175980975029 and parameters: {'lr': 1.0982356207139913e-05, 'weight_decay': 3.1698429943672157e-05}. Best is trial 0 with value: 0.07372175980975029.
/var/folders/4g/8j_z4q0n2zlftk4s8jmrj1sw0000gn/T/ipykernel_36442/2510586555.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform('lr', 1e-5, 1e-3)
/var/folders/4g/8j_z4q0n2zlftk4s8jmrj1sw0000gn/T/ipykernel_36442/2510586555.py:6: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
/Users/nanphattongsirisukool/Documents/GitHub/Structural-Defects-Network-MLOps/env/lib/pytho

Epoch 1/3 | Train Loss: 0.2542 | Train Acc: 0.7496 | Val Loss: 0.4307 | Val F1: 0.6016
Epoch 2/3 | Train Loss: 0.1729 | Train Acc: 0.8367 | Val Loss: 0.3798 | Val F1: 0.6446
Epoch 3/3 | Train Loss: 0.1383 | Train Acc: 0.8718 | Val Loss: 0.3152 | Val F1: 0.7093


...

Epoch 1/3 | Train Loss: 0.2384 | Train Acc: 0.7558 | Val Loss: 0.5502 | Val F1: 0.4602
Epoch 2/3 | Train Loss: 0.1907 | Train Acc: 0.8149 | Val Loss: 0.4143 | Val F1: 0.6010
Epoch 3/3 | Train Loss: 0.1728 | Train Acc: 0.8316 | Val Loss: 0.4559 | Val F1: 0.5587


[I 2025-05-20 01:07:37,296] Trial 18 finished with value: 0.07372175980975029 and parameters: {'lr': 0.00010249755290289741, 'weight_decay': 1.1927780767639551e-05}. Best is trial 0 with value: 0.07372175980975029.
/var/folders/4g/8j_z4q0n2zlftk4s8jmrj1sw0000gn/T/ipykernel_36442/2510586555.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform('lr', 1e-5, 1e-3)
/var/folders/4g/8j_z4q0n2zlftk4s8jmrj1sw0000gn/T/ipykernel_36442/2510586555.py:6: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
/Users/nanphattongsirisukool/Documents/GitHub/Structural-Defects-Network-MLOps/env/lib/pyth

Epoch 1/3 | Train Loss: 0.2707 | Train Acc: 0.7280 | Val Loss: 0.4091 | Val F1: 0.6336
Epoch 2/3 | Train Loss: 0.1787 | Train Acc: 0.8349 | Val Loss: 0.3132 | Val F1: 0.7254
Epoch 3/3 | Train Loss: 0.1436 | Train Acc: 0.8685 | Val Loss: 0.3503 | Val F1: 0.6814


[I 2025-05-20 02:39:39,021] Trial 19 finished with value: 0.07372175980975029 and parameters: {'lr': 1.81100042304994e-05, 'weight_decay': 2.9698338762073575e-06}. Best is trial 0 with value: 0.07372175980975029.


Best trial:
FrozenTrial(number=0, state=1, values=[0.07372175980975029], datetime_start=datetime.datetime(2025, 5, 18, 4, 16, 30, 469831), datetime_complete=datetime.datetime(2025, 5, 18, 5, 38, 42, 904685), params={'lr': 1.0982356207139913e-05, 'weight_decay': 3.1698429943672157e-05}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'lr': FloatDistribution(high=0.001, log=True, low=1e-05, step=None), 'weight_decay': FloatDistribution(high=0.001, log=True, low=1e-06, step=None)}, trial_id=0, value=None)


In [13]:
print("Best trial:")
print(study.best_trial)

Best trial:
FrozenTrial(number=0, state=1, values=[0.07372175980975029], datetime_start=datetime.datetime(2025, 5, 18, 4, 16, 30, 469831), datetime_complete=datetime.datetime(2025, 5, 18, 5, 38, 42, 904685), params={'lr': 1.0982356207139913e-05, 'weight_decay': 3.1698429943672157e-05}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'lr': FloatDistribution(high=0.001, log=True, low=1e-05, step=None), 'weight_decay': FloatDistribution(high=0.001, log=True, low=1e-06, step=None)}, trial_id=0, value=None)


In [14]:
best_params = study.best_params
optimizer = optim.Adam(model.parameters(),
                       lr=best_params['lr'],
                       weight_decay=best_params['weight_decay'])



In [15]:
import json

# Save best hyperparameters
with open("best_params_6040.json", "w") as f:
    json.dump(best_params, f, indent=4)


In [16]:
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model.to(device)


/Users/nanphattongsirisukool/Documents/GitHub/Structural-Defects-Network-MLOps/env/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/nanphattongsirisukool/Documents/GitHub/Structural-Defects-Network-MLOps/env/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [17]:
criterion = SoftF1Loss()


In [18]:
optimizer = optim.Adam(model.parameters(),
                       lr=best_params['lr'],
                       weight_decay=best_params['weight_decay'])


In [19]:
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)


In [20]:
train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=10)

# Epoch 1/10 | Train Loss: 0.2800 | Train Acc: 0.7329 | Val Loss: 0.4155 | Val F1: 0.6282
# Epoch 2/10 | Train Loss: 0.1851 | Train Acc: 0.8300 | Val Loss: 0.3592 | Val F1: 0.6721
# Epoch 3/10 | Train Loss: 0.1483 | Train Acc: 0.8649 | Val Loss: 0.3262 | Val F1: 0.7126
# Epoch 4/10 | Train Loss: 0.1231 | Train Acc: 0.8904 | Val Loss: 0.3482 | Val F1: 0.6828
# Epoch 5/10 | Train Loss: 0.1026 | Train Acc: 0.9082 | Val Loss: 0.3205 | Val F1: 0.7067
# Epoch 6/10 | Train Loss: 0.0837 | Train Acc: 0.9267 | Val Loss: 0.2913 | Val F1: 0.7425
# Epoch 7/10 | Train Loss: 0.0715 | Train Acc: 0.9369 | Val Loss: 0.2750 | Val F1: 0.7583
# Epoch 8/10 | Train Loss: 0.0630 | Train Acc: 0.9459 | Val Loss: 0.3046 | Val F1: 0.7289
# Epoch 9/10 | Train Loss: 0.0566 | Train Acc: 0.9514 | Val Loss: 0.2897 | Val F1: 0.7416
# Epoch 10/10 | Train Loss: 0.0492 | Train Acc: 0.9572 | Val Loss: 0.2939 | Val F1: 0.7347


Epoch 1/10 | Train Loss: 0.3228 | Train Acc: 0.6881 | Val Loss: 0.4646 | Val F1: 0.6083
Epoch 2/10 | Train Loss: 0.2084 | Train Acc: 0.8111 | Val Loss: 0.3896 | Val F1: 0.6576
Epoch 3/10 | Train Loss: 0.1683 | Train Acc: 0.8467 | Val Loss: 0.3421 | Val F1: 0.7002
Epoch 4/10 | Train Loss: 0.1394 | Train Acc: 0.8755 | Val Loss: 0.3297 | Val F1: 0.7058
Epoch 5/10 | Train Loss: 0.1198 | Train Acc: 0.8947 | Val Loss: 0.3228 | Val F1: 0.7149
Epoch 6/10 | Train Loss: 0.1020 | Train Acc: 0.9116 | Val Loss: 0.3223 | Val F1: 0.7144
Epoch 7/10 | Train Loss: 0.0835 | Train Acc: 0.9284 | Val Loss: 0.3592 | Val F1: 0.6703
Epoch 8/10 | Train Loss: 0.0743 | Train Acc: 0.9358 | Val Loss: 0.2798 | Val F1: 0.7541
Epoch 9/10 | Train Loss: 0.0608 | Train Acc: 0.9482 | Val Loss: 0.3547 | Val F1: 0.6770
Epoch 10/10 | Train Loss: 0.0544 | Train Acc: 0.9553 | Val Loss: 0.3125 | Val F1: 0.7204


In [ ]:
torch.save(model.state_dict(), '../model/resnet_best_params_6040.pth')

In [21]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

# Make sure model is in eval mode
model.eval()

# Consistent transform (same used during training)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Evaluation function
def evaluate_model_on_split(split_name):
    base_path = f"../artifact_folder/{split_name}"
    labels_df = pd.read_csv(os.path.join(base_path, "labels.csv"))
    images_dir = os.path.join(base_path, "images")

    y_true, y_pred, y_score = [], [], []

    for _, row in labels_df.iterrows():
        img_path = os.path.join(images_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")
        input_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            pred = output.argmax(dim=1).item()
            prob = torch.softmax(output, dim=1)[0][1].item()

        label = 1 if row["label"].lower() == "cracked" else 0
        y_true.append(label)
        y_pred.append(pred)
        y_score.append(prob)

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "AUC-ROC": roc_auc_score(y_true, y_score),
        "Confusion Matrix": confusion_matrix(y_true, y_pred).tolist()
    }

# Run on all sets
for split in ['train', 'val', 'test']:
    print(f"📊 Evaluation for {split.upper()}")
    metrics = evaluate_model_on_split(split)
    for k, v in metrics.items():
        print(f"{k}: {v}")
    print()


📊 Evaluation for TRAIN
Accuracy: 0.9681164545025931
Precision: 0.9774011299435028
Recall: 0.9583922677982084
F1-Score: 0.9678033684461108
AUC-ROC: 0.979215264676599
Confusion Matrix: [[8296, 188], [353, 8131]]

📊 Evaluation for VAL
Accuracy: 0.9013431593961726
Precision: 0.6340450771055753
Recall: 0.8338533541341654
F1-Score: 0.7203504043126685
AUC-ROC: 0.9285018434813961
Confusion Matrix: [[6514, 617], [213, 1069]]

📊 Evaluation for TEST
Accuracy: 0.9023172905525847
Precision: 0.6308328411104548
Recall: 0.8442687747035573
F1-Score: 0.7221095334685599
AUC-ROC: 0.9407609939467647
Confusion Matrix: [[6525, 625], [197, 1068]]



In [68]:
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model.to(device)
criterion = SoftF1Loss()
optimizer = optim.Adam(model.parameters(),
                       lr=best_params['lr'],
                       weight_decay=best_params['weight_decay'])
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)


/Users/nanphattongsirisukool/Documents/GitHub/Structural-Defects-Network-MLOps/env/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/nanphattongsirisukool/Documents/GitHub/Structural-Defects-Network-MLOps/env/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [69]:
model = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=30, patience=4)


Epoch 1/30 | Train Loss: 0.2809 | Train Acc: 0.7398 | Val Loss: 0.3806 | Val F1: 0.6784
Epoch 2/30 | Train Loss: 0.1840 | Train Acc: 0.8302 | Val Loss: 0.3103 | Val F1: 0.7273
Epoch 3/30 | Train Loss: 0.1491 | Train Acc: 0.8640 | Val Loss: 0.3165 | Val F1: 0.7191
Epoch 4/30 | Train Loss: 0.1194 | Train Acc: 0.8940 | Val Loss: 0.3511 | Val F1: 0.6830
Epoch 5/30 | Train Loss: 0.1012 | Train Acc: 0.9100 | Val Loss: 0.2776 | Val F1: 0.7559
Epoch 6/30 | Train Loss: 0.0806 | Train Acc: 0.9300 | Val Loss: 0.3680 | Val F1: 0.6633
Epoch 7/30 | Train Loss: 0.0685 | Train Acc: 0.9412 | Val Loss: 0.3110 | Val F1: 0.7200
Epoch 8/30 | Train Loss: 0.0579 | Train Acc: 0.9501 | Val Loss: 0.3094 | Val F1: 0.7195
Epoch 9/30 | Train Loss: 0.0459 | Train Acc: 0.9616 | Val Loss: 0.2978 | Val F1: 0.7339
⏹️ Early stopping triggered at epoch 9. Best Val F1: 0.7559
